# Thesis Dataset Analysis: A3 vs DMF5

> Goal: generate publication-ready TCR analyses for two datasets with reproducible settings and runtime controls.

This notebook compares two datasets and produces one combined result figure with five rows and two columns:
1. WT-distance distribution with the average Hamming distance marked for each dataset.
2. Within-dataset pairwise TCR distance distribution (sampled for runtime control).
3. Cross-reactive binder depth: number of TCRs that bind (`label = 1`) to 1, 2, 3, ... peptides.
4. Peptide-level single-binder label context: for each positive peptide, TCRs with one positive peptide and either at least one observed negative peptide elsewhere or no observations for all other peptides.
5. TCR label multiplicity: all loaded TCRs with one observed peptide-label entry versus multi-label TCRs split into all-0, all-1, and mixed 0/1 groups.

Current settings are configured for:
- A3: `/cluster/project/reddy/katja/ml_refactor/data/data_from_NGS_pipline/A3_Q20_3x1x_tcr_peptide_label_all.csv`
- DMF5: `/cluster/project/reddy/katja/ml_refactor/data/data_from_NGS_pipline/DMF5_Q30_2x_tcr_peptide_label_all.csv`
- WT TCRs: `ASSPNMADEQY` (A3), `ASSLSFGTEA` (DMF5)
- Analysis preview subsampling: `N = 5,000` unique TCRs per dataset for rows 1-4; row 5 label multiplicity uses all loaded TCRs; pairwise-distance preview subsampling: `N = 250` unique TCRs per dataset

In [1]:
# Imports and global plotting style
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
sns.set_context('talk')
np.random.seed(42)

In [2]:
# Configuration: dataset paths, WT sequences, runtime controls, output folder
dataset_config = {
    'A3': {
        'path': '/cluster/project/reddy/katja/ml_refactor/data/data_from_NGS_pipline/A3_Q20_3x1x_tcr_peptide_label_all.csv',
        'wt_tcr': 'ASSPNMADEQY',
    },
    'DMF5': {
        'path': '/cluster/project/reddy/katja/ml_refactor/data/data_from_NGS_pipline/DMF5_Q30_2x_tcr_peptide_label_all.csv',
        'wt_tcr': 'ASSLSFGTEAF',
    },
}

required_columns = ['tcr', 'peptide', 'label']
subsample_n = 250
n_random_pairs = 20_000
random_seed = 42

# Preview controls: keep these small while tuning plot layout; set analysis_tcr_subsample_n to None for full export.
analysis_tcr_subsample_n = 5000
analysis_subsample_seed = 42

output_dir = Path('/cluster/project/reddy/katja/NGS_pipeline/results/thesis_dataset_analysis')
output_dir.mkdir(parents=True, exist_ok=True)

print('Output directory:', output_dir)
print('Datasets configured:', ', '.join(dataset_config.keys()))
print('Analysis TCR subsample per dataset:', analysis_tcr_subsample_n)

Output directory: /cluster/project/reddy/katja/NGS_pipeline/results/thesis_dataset_analysis
Datasets configured: A3, DMF5
Analysis TCR subsample per dataset: 5000


In [3]:
# Load both datasets and validate schema
datasets = {}
for name, cfg in dataset_config.items():
    path = Path(cfg['path'])
    if not path.exists():
        raise FileNotFoundError(f'[{name}] File not found: {path}')

    df = pd.read_csv(path)
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f'[{name}] Missing required columns: {missing}')

    df = df.copy()
    df['tcr'] = df['tcr'].astype(str).str.strip().str.upper()
    df['peptide'] = df['peptide'].astype(str).str.strip()
    df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)

    datasets[name] = df
    print(f'[{name}] loaded {len(df):,} rows | unique TCRs={df["tcr"].nunique():,} | unique peptides={df["peptide"].nunique():,}')

print('\nPreview:')
for name, df in datasets.items():
    display(df.head(3))

analysis_datasets = {}
for name, df in datasets.items():
    if analysis_tcr_subsample_n is None:
        analysis_datasets[name] = df
        print(f'[{name}] analysis uses full dataset')
    else:
        sampled_tcrs = (
            df['tcr']
            .drop_duplicates()
            .sample(n=min(analysis_tcr_subsample_n, df['tcr'].nunique()), random_state=analysis_subsample_seed)
        )
        analysis_datasets[name] = df[df['tcr'].isin(sampled_tcrs)].copy()
        print(
            f'[{name}] analysis preview subset: {len(analysis_datasets[name]):,} rows | '
            f'unique TCRs={analysis_datasets[name]["tcr"].nunique():,}'
        )


[A3] loaded 431,843 rows | unique TCRs=156,120 | unique peptides=10
[DMF5] loaded 431,695 rows | unique TCRs=203,134 | unique peptides=6

Preview:


,tcr,peptide,label
0,AGGANAADEQF,EGDPLILQY,0
1,AGGANAADGLY,EGDPLILQY,0
2,AGGANAADGQF,EGDPLILQY,0


,tcr,peptide,label
0,AAALGFGTEAF,IMEDVGWLNV,0
1,AAALRFGTEAF,IMEDVGWLNV,0
2,AAALSFGGEAF,IMEDVGWLNV,0


[A3] analysis preview subset: 13,894 rows | unique TCRs=5,000
[DMF5] analysis preview subset: 10,475 rows | unique TCRs=5,000


In [4]:
# Analysis helpers for WT distance, cross-reactivity depth, label multiplicity, and sampled pairwise distances
def hamming_distance_equal_len(a: str, b: str) -> int:
    if len(a) != len(b):
        raise ValueError('Hamming distance requires equal-length strings')
    return sum(x != y for x, y in zip(a, b))


def compute_wt_distance_table(df: pd.DataFrame, wt_tcr: str) -> tuple[pd.DataFrame, dict]:
    unique_tcr_df = df[['tcr']].drop_duplicates().copy()
    wt_tcr = wt_tcr.strip().upper()
    wt_len = len(wt_tcr)

    unique_tcr_df['len'] = unique_tcr_df['tcr'].str.len()
    valid = unique_tcr_df[unique_tcr_df['len'] == wt_len].copy()
    invalid = unique_tcr_df[unique_tcr_df['len'] != wt_len].copy()

    valid['wt_hamming_distance'] = valid['tcr'].apply(lambda s: hamming_distance_equal_len(s, wt_tcr))

    # row-weighted variant for reporting
    row_level = df[df['tcr'].isin(valid['tcr'])].copy()
    row_level = row_level.merge(valid[['tcr', 'wt_hamming_distance']], on='tcr', how='left')

    stats = {
        'wt_length': wt_len,
        'unique_tcr_total': int(unique_tcr_df['tcr'].nunique()),
        'unique_tcr_valid_len': int(valid['tcr'].nunique()),
        'unique_tcr_invalid_len': int(invalid['tcr'].nunique()),
        'mean_distance_unique_tcr': float(valid['wt_hamming_distance'].mean()) if len(valid) else np.nan,
        'median_distance_unique_tcr': float(valid['wt_hamming_distance'].median()) if len(valid) else np.nan,
        'mean_distance_row_weighted': float(row_level['wt_hamming_distance'].mean()) if len(row_level) else np.nan,
    }
    return valid, stats


def compute_crossreactive_binder_depth(df: pd.DataFrame) -> pd.DataFrame:
    binders = df[df['label'] == 1].copy()
    if binders.empty:
        return pd.DataFrame(columns=['n_bound_peptides', 'n_tcrs'])

    # Include depth 1 to show TCRs that bind only one peptide, plus depth >= 2 cross-reactive TCRs.
    depth = binders.groupby('tcr')['peptide'].nunique()
    depth = depth[depth >= 1]
    if depth.empty:
        return pd.DataFrame(columns=['n_bound_peptides', 'n_tcrs'])

    out = (
        depth.value_counts()
        .sort_index()
        .rename_axis('n_bound_peptides')
        .reset_index(name='n_tcrs')
    )
    return out


def compute_single_binder_label_context(df: pd.DataFrame, all_peptides: list[str] | None = None) -> pd.DataFrame:
    if all_peptides is None:
        all_peptides = sorted(df['peptide'].dropna().unique())
    n_peptides_total = len(all_peptides)

    peptide_labels = (
        df.groupby(['tcr', 'peptide'], as_index=False)['label']
        .max()
    )
    per_tcr = peptide_labels.groupby('tcr').agg(
        n_bound_peptides=('label', lambda s: int((s == 1).sum())),
        n_negative_peptides=('label', lambda s: int((s == 0).sum())),
        n_observed_peptides=('peptide', 'nunique'),
    )
    per_tcr = per_tcr[per_tcr['n_bound_peptides'] == 1].copy()
    if per_tcr.empty:
        return pd.DataFrame(columns=['single_binder_context', 'n_tcrs'])

    per_tcr['single_binder_context'] = np.where(
        per_tcr['n_negative_peptides'] > 0,
        '1 positive + >=1 negative',
        '1 positive + N/A for all others',
    )
    per_tcr['n_unobserved_peptides'] = n_peptides_total - per_tcr['n_observed_peptides']

    out = (
        per_tcr['single_binder_context']
        .value_counts()
        .reindex(['1 positive + >=1 negative', '1 positive + N/A for all others'], fill_value=0)
        .rename_axis('single_binder_context')
        .reset_index(name='n_tcrs')
    )
    return out


def compute_single_binder_label_context_by_peptide(df: pd.DataFrame, all_peptides: list[str] | None = None) -> pd.DataFrame:
    if all_peptides is None:
        all_peptides = sorted(df['peptide'].dropna().unique())

    peptide_labels = df.groupby(['tcr', 'peptide'], as_index=False)['label'].max()
    positive_peptides = (
        peptide_labels[peptide_labels['label'] == 1]
        .groupby('tcr')['peptide']
        .agg(list)
        .rename('positive_peptides')
    )
    negative_counts = (
        peptide_labels[peptide_labels['label'] == 0]
        .groupby('tcr')['peptide']
        .nunique()
        .rename('n_negative_peptides')
    )
    per_tcr = positive_peptides.to_frame().join(negative_counts, how='left')
    per_tcr['n_negative_peptides'] = per_tcr['n_negative_peptides'].fillna(0).astype(int)
    per_tcr['n_bound_peptides'] = per_tcr['positive_peptides'].str.len()
    per_tcr = per_tcr[per_tcr['n_bound_peptides'] == 1].copy()

    if per_tcr.empty:
        return pd.DataFrame(columns=['peptide', 'single_binder_context', 'n_tcrs'])

    per_tcr['peptide'] = per_tcr['positive_peptides'].str[0]
    per_tcr['single_binder_context'] = np.where(
        per_tcr['n_negative_peptides'] > 0,
        'Positive on one peptide, only 0 on other peptides',
        'Positive on one peptide, absent elsewhere',
    )

    context_order = [
        'Positive on one peptide, absent elsewhere',
        'Positive on one peptide, only 0 on other peptides',
    ]
    full_index = pd.MultiIndex.from_product(
        [all_peptides, context_order],
        names=['peptide', 'single_binder_context'],
    )
    out = (
        per_tcr.groupby(['peptide', 'single_binder_context'])
        .size()
        .reindex(full_index, fill_value=0)
        .rename('n_tcrs')
        .reset_index()
    )
    return out



def compute_tcr_label_multiplicity(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    peptide_labels = (
        df[['tcr', 'peptide', 'label']]
        .groupby(['tcr', 'peptide'], sort=False, as_index=False)['label']
        .max()
    )
    per_tcr = peptide_labels.groupby('tcr', sort=False)['label'].agg(
        n_observed_labels='size',
        min_label='min',
        max_label='max',
    )

    single_count = int((per_tcr['n_observed_labels'] == 1).sum())
    multiple_mask = per_tcr['n_observed_labels'] > 1
    multiple_all_0 = int((multiple_mask & (per_tcr['max_label'] == 0)).sum())
    multiple_all_1 = int((multiple_mask & (per_tcr['min_label'] == 1)).sum())
    multiple_mixed = int((multiple_mask & (per_tcr['min_label'] == 0) & (per_tcr['max_label'] == 1)).sum())

    counts = pd.DataFrame({
        'label_multiplicity_group': [
            'Single observed label',
            'Multiple all 0',
            'Multiple all 1',
            'Multiple mixed 0/1',
        ],
        'n_tcrs': [
            single_count,
            multiple_all_0,
            multiple_all_1,
            multiple_mixed,
        ],
    })
    summary = pd.DataFrame({
        'label_multiplicity_summary': ['Single observed label', 'Multiple observed labels'],
        'n_tcrs': [single_count, multiple_all_0 + multiple_all_1 + multiple_mixed],
    })
    return counts, summary

def sample_pairwise_hamming_distances(df: pd.DataFrame, sample_n: int, n_pairs: int, seed: int) -> tuple[np.ndarray, dict]:
    unique_tcrs = pd.Index(df['tcr'].dropna().astype(str).str.strip().str.upper().unique())
    if len(unique_tcrs) < 2:
        return np.array([], dtype=int), {
            'unique_tcr_total': int(len(unique_tcrs)),
            'sampled_unique_tcrs': int(len(unique_tcrs)),
            'pairs_computed': 0,
        }

    lengths = pd.Series(unique_tcrs).str.len()
    mode_len = int(lengths.mode().iat[0])
    same_len_mask = lengths == mode_len
    candidate_tcrs = unique_tcrs[same_len_mask.to_numpy()]

    rng = np.random.default_rng(seed)
    k = min(sample_n, len(candidate_tcrs))
    sampled = rng.choice(candidate_tcrs.to_numpy(), size=k, replace=False)

    arr = np.array([list(seq) for seq in sampled])
    n = len(arr)
    if n < 2:
        return np.array([], dtype=int), {
            'unique_tcr_total': int(len(unique_tcrs)),
            'candidate_equal_len_tcrs': int(len(candidate_tcrs)),
            'sampled_unique_tcrs': int(n),
            'pairs_computed': 0,
            'sequence_length_used': mode_len,
        }

    i = rng.integers(0, n, size=n_pairs)
    j = rng.integers(0, n, size=n_pairs)
    mask = i != j
    i = i[mask]
    j = j[mask]

    dists = np.sum(arr[i] != arr[j], axis=1)

    stats = {
        'unique_tcr_total': int(len(unique_tcrs)),
        'candidate_equal_len_tcrs': int(len(candidate_tcrs)),
        'sampled_unique_tcrs': int(n),
        'pairs_requested': int(n_pairs),
        'pairs_computed': int(len(dists)),
        'sequence_length_used': int(mode_len),
        'mean_distance': float(np.mean(dists)) if len(dists) else np.nan,
        'median_distance': float(np.median(dists)) if len(dists) else np.nan,
        'p05': float(np.percentile(dists, 5)) if len(dists) else np.nan,
        'p95': float(np.percentile(dists, 95)) if len(dists) else np.nan,
    }
    return dists, stats

In [ ]:
# Run analyses for both datasets and collect tidy outputs
wt_distance_frames = []
crossreact_frames = []
pairwise_frames = []
single_binder_context_frames = []
single_binder_context_by_peptide_frames = []
label_multiplicity_frames = []
label_multiplicity_summary_frames = []
peptide_orders = {}
validation_rows = []

for name, df in analysis_datasets.items():
    wt_table, wt_stats = compute_wt_distance_table(df, dataset_config[name]['wt_tcr'])
    wt_table['dataset'] = name
    wt_distance_frames.append(wt_table[['dataset', 'tcr', 'wt_hamming_distance']])

    cr = compute_crossreactive_binder_depth(df)
    cr['dataset'] = name
    crossreact_frames.append(cr[['dataset', 'n_bound_peptides', 'n_tcrs']])

    peptide_order = sorted(datasets[name]['peptide'].dropna().unique())
    peptide_orders[name] = peptide_order
    single_ctx = compute_single_binder_label_context(df, peptide_order)
    single_ctx['dataset'] = name
    single_binder_context_frames.append(single_ctx[['dataset', 'single_binder_context', 'n_tcrs']])

    single_ctx_by_peptide = compute_single_binder_label_context_by_peptide(df, peptide_order)
    single_ctx_by_peptide['dataset'] = name
    single_binder_context_by_peptide_frames.append(single_ctx_by_peptide[['dataset', 'peptide', 'single_binder_context', 'n_tcrs']])

    # Use the full dataset for label multiplicity, even when rows 1-4 use the preview subset.
    label_df = datasets[name]
    label_mult, label_mult_summary = compute_tcr_label_multiplicity(label_df)
    label_mult['dataset'] = name
    label_multiplicity_frames.append(label_mult[['dataset', 'label_multiplicity_group', 'n_tcrs']])
    label_mult_summary['dataset'] = name
    label_multiplicity_summary_frames.append(label_mult_summary[['dataset', 'label_multiplicity_summary', 'n_tcrs']])

    dists, dist_stats = sample_pairwise_hamming_distances(
        df=df,
        sample_n=subsample_n,
        n_pairs=n_random_pairs,
        seed=random_seed,
    )
    pairwise_frames.append(pd.DataFrame({'dataset': name, 'pairwise_hamming_distance': dists}))

    validation_rows.append({
        'dataset': name,
        **wt_stats,
        **dist_stats,
        'rows_analyzed': int(len(df)),
        'unique_peptides_total': int(len(peptide_order)),
        'binder_rows': int((df['label'] == 1).sum()),
        'crossreactive_binder_tcrs': int(cr.loc[cr['n_bound_peptides'] >= 2, 'n_tcrs'].sum()) if not cr.empty else 0,
        'single_binder_context_tcrs': int(single_ctx['n_tcrs'].sum()) if not single_ctx.empty else 0,
        'label_multiplicity_unique_tcrs': int(label_df['tcr'].nunique()),
        'label_multiplicity_single_tcrs': int(label_mult.loc[label_mult['label_multiplicity_group'] == 'Single observed label', 'n_tcrs'].sum()),
        'label_multiplicity_multiple_tcrs': int(label_mult_summary.loc[label_mult_summary['label_multiplicity_summary'] == 'Multiple observed labels', 'n_tcrs'].sum()),
        'label_multiplicity_multiple_all_0_tcrs': int(label_mult.loc[label_mult['label_multiplicity_group'] == 'Multiple all 0', 'n_tcrs'].sum()),
        'label_multiplicity_multiple_all_1_tcrs': int(label_mult.loc[label_mult['label_multiplicity_group'] == 'Multiple all 1', 'n_tcrs'].sum()),
        'label_multiplicity_multiple_mixed_tcrs': int(label_mult.loc[label_mult['label_multiplicity_group'] == 'Multiple mixed 0/1', 'n_tcrs'].sum()),
    })

wt_distance_df = pd.concat(wt_distance_frames, ignore_index=True)
crossreact_df = pd.concat(crossreact_frames, ignore_index=True)
pairwise_df = pd.concat(pairwise_frames, ignore_index=True)
single_binder_context_df = pd.concat(single_binder_context_frames, ignore_index=True)
single_binder_context_by_peptide_df = pd.concat(single_binder_context_by_peptide_frames, ignore_index=True)
label_multiplicity_df = pd.concat(label_multiplicity_frames, ignore_index=True)
label_multiplicity_summary_df = pd.concat(label_multiplicity_summary_frames, ignore_index=True)
validation_df = pd.DataFrame(validation_rows)

# Full datasets are no longer needed after row 5 counts are computed.
del label_df
del datasets

display(validation_df)

In [ ]:
# Combined final figure: 5 plot types x 2 datasets
# Rows: WT-distance distribution, pairwise TCR-distance distribution, cross-reactive binder depth, single-binder label context, full-dataset TCR label multiplicity.
# Columns: A3 and DMF5.
from matplotlib.ticker import FuncFormatter


def format_thousands_as_k(x, pos):
    if abs(x) >= 1000:
        value = x / 1000
        return f'{value:.0f}K' if value.is_integer() else f'{value:.1f}K'
    return f'{x:.0f}'


def annotate_bars(ax, bars, values, formatter=lambda v: f'{int(v):,}'):
    if len(values) == 0:
        return
    ymax = max(values)
    offset = max(ymax * 0.015, 1)
    for bar, value in zip(bars, values):
        if value <= 0:
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            formatter(value),
            ha='center',
            va='bottom',
            fontsize=10,
            rotation=0,
            clip_on=False,
        )


def style_axis(ax):
    ax.tick_params(axis='both', labelsize=TICK_LABEL_SIZE)
    ax.grid(False)
    ax.set_facecolor('white')


datasets_order = ['A3', 'DMF5']
datasets_order = [d for d in datasets_order if d in analysis_datasets]
class_specs = [
    ('specific', 'specific', ['#B7E4C7', '#74C69D', '#40916C', '#2D6A4F', '#1B4332']),
    ('unspecific', 'unspecific', ['#BFD7EA', '#76A7C8', '#3F7FA8', '#1F5A7A', '#12384F']),
]

# A3 uses the green specific palette; DMF5 uses the blue unspecific palette.
color_map = {
    'A3': class_specs[0][2][2],
    'DMF5': class_specs[1][2][2],
}
count_formatter = FuncFormatter(format_thousands_as_k)

TITLE_SIZE = 20
AXIS_LABEL_SIZE = 15
TICK_LABEL_SIZE = 12
ROW_LABEL_SIZE = 17
LEGEND_SIZE = 12
SUPTITLE_SIZE = 24
BAR_WIDTH = 0.78
HIST_RWIDTH = 0.88

if not datasets_order:
    raise ValueError('No configured datasets are available for plotting.')

wt_max_distance = 0
for dataset_name in datasets_order:
    wt = dataset_config[dataset_name]['wt_tcr'].strip().upper()
    wt_max_distance = max(wt_max_distance, len(wt))

pairwise_nonempty = pairwise_df['pairwise_hamming_distance'].dropna()
if pairwise_nonempty.empty:
    pairwise_min_distance, pairwise_max_distance = 0, 1
else:
    pairwise_min_distance = int(np.floor(pairwise_nonempty.min()))
    pairwise_max_distance = int(np.ceil(pairwise_nonempty.max()))

crossreact_max_depth = int(crossreact_df['n_bound_peptides'].max()) if not crossreact_df.empty else 1
single_context_order = [
    'Positive on one peptide, absent elsewhere',
    'Positive on one peptide, only 0 on other peptides',
]
single_context_legend_labels = [
    'Positive on one peptide, absent elsewhere',
    'Positive on one peptide, only 0 on other peptides',
]
label_multiplicity_order = [
    'Single observed label',
    'Multiple all 0',
    'Multiple all 1',
    'Multiple mixed 0/1',
]
label_multiplicity_xticklabels = [
    'Single\nobserved\nlabel',
    'Multiple\nall 0',
    'Multiple\nall 1',
    'Multiple\nmixed\n0/1',
]
max_peptides_per_dataset = max(
    len(peptide_orders[name])
    for name in datasets_order
)

plt.rcdefaults()
sns.set_style('white', {'axes.grid': False})
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'Liberation Serif', 'DejaVu Serif'],
})
fig, axes = plt.subplots(
    5,
    len(datasets_order),
    figsize=(7.5 * len(datasets_order), 28.0),
    facecolor='white',
    squeeze=False,
)

for col_idx, dataset_name in enumerate(datasets_order):
    dataset_color = color_map.get(dataset_name, '#666666')

    # Row 1: WT-distance distribution with average marked
    ax = axes[0, col_idx]
    df_curr = analysis_datasets[dataset_name]
    wt = dataset_config[dataset_name]['wt_tcr'].strip().upper()
    wt_len = len(wt)
    tcr_series = df_curr['tcr'].astype(str).str.strip().str.upper()
    valid = tcr_series[tcr_series.str.len() == wt_len]
    if len(valid) == 0:
        ax.text(0.5, 0.5, 'No valid-length TCRs', ha='center', va='center', transform=ax.transAxes, fontsize=AXIS_LABEL_SIZE)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        row_dists = valid.apply(lambda seq: hamming_distance_equal_len(seq, wt))
        distance_bins = np.arange(0, wt_max_distance + 1, dtype=int)
        counts = row_dists.value_counts().reindex(distance_bins, fill_value=0)
        xvals = distance_bins
        yvals = counts.to_numpy()
        mean_dist = float(row_dists.mean())
        ax.bar(xvals, yvals, color=dataset_color, edgecolor='black', linewidth=0.4, width=BAR_WIDTH)
        ax.axvline(mean_dist, color='black', linestyle='--', linewidth=1.6, label=f'Mean = {mean_dist:.2f}')
        ax.legend(frameon=False, loc='upper right', fontsize=LEGEND_SIZE)
        ax.set_xticks(distance_bins)
        ax.set_xlim(-0.5, wt_max_distance + 0.5)
        ax.set_ylim(0, max(10, int(yvals.max() * 1.08) + 1))
        ax.yaxis.set_major_formatter(count_formatter)
    ax.set_title(dataset_name, fontsize=TITLE_SIZE, fontweight='bold', pad=30)
    ax.set_xlabel('Hamming distance to WT TCR', fontsize=AXIS_LABEL_SIZE)
    if col_idx == 0:
        ax.set_ylabel('Number of TCR rows', fontsize=AXIS_LABEL_SIZE)
    style_axis(ax)

    # Row 2: Sampled within-dataset pairwise TCR distance distribution
    ax = axes[1, col_idx]
    subset = pairwise_df[pairwise_df['dataset'] == dataset_name]['pairwise_hamming_distance'].dropna().to_numpy()
    if len(subset) == 0:
        ax.text(0.5, 0.5, 'No pairwise distances', ha='center', va='center', transform=ax.transAxes, fontsize=AXIS_LABEL_SIZE)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        bins = np.arange(pairwise_min_distance, pairwise_max_distance + 2) - 0.5
        ax.hist(
            subset,
            bins=bins,
            density=True,
            color=dataset_color,
            edgecolor='black',
            linewidth=0.4,
            alpha=0.9,
            rwidth=HIST_RWIDTH,
        )
        ax.set_xticks(np.arange(pairwise_min_distance, pairwise_max_distance + 1))
        ax.set_xlim(pairwise_min_distance - 0.5, pairwise_max_distance + 0.5)
    ax.set_xlabel('Pairwise Hamming distance', fontsize=AXIS_LABEL_SIZE)
    if col_idx == 0:
        ax.set_ylabel('Density', fontsize=AXIS_LABEL_SIZE)
    style_axis(ax)

    # Row 3: Cross-reactive binder depth
    ax = axes[2, col_idx]
    subset = crossreact_df[crossreact_df['dataset'] == dataset_name].copy()
    subset = subset.sort_values('n_bound_peptides')
    if subset.empty:
        ax.text(0.5, 0.5, 'No binders', ha='center', va='center', transform=ax.transAxes, fontsize=AXIS_LABEL_SIZE)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        depth_bins = np.arange(1, crossreact_max_depth + 1, dtype=int)
        counts_by_depth = subset.set_index('n_bound_peptides')['n_tcrs']
        xvals = depth_bins
        yvals = np.array([counts_by_depth.get(depth, 0) for depth in depth_bins])
        bars = ax.bar(xvals, yvals, color=dataset_color, edgecolor='black', linewidth=0.4, width=BAR_WIDTH)
        annotate_bars(ax, bars, yvals)
        ax.set_xticks(xvals)
        ax.set_xlim(0.5, crossreact_max_depth + 0.5)
        ax.set_ylim(0, max(5, int(yvals.max() * 1.22) + 1))
        ax.yaxis.set_major_formatter(count_formatter)
    ax.set_xlabel('Peptides bound per TCR', fontsize=AXIS_LABEL_SIZE)
    if col_idx == 0:
        ax.set_ylabel('Number of TCRs', fontsize=AXIS_LABEL_SIZE)
    style_axis(ax)

    # Row 4: Peptide-level single-binder label context
    ax = axes[3, col_idx]
    peptide_order = peptide_orders[dataset_name]
    subset = single_binder_context_by_peptide_df[
        single_binder_context_by_peptide_df['dataset'] == dataset_name
    ].copy()
    pivot = (
        subset.pivot_table(
            index='peptide',
            columns='single_binder_context',
            values='n_tcrs',
            aggfunc='sum',
            fill_value=0,
        )
        .reindex(peptide_order, fill_value=0)
        .reindex(columns=single_context_order, fill_value=0)
    )
    x_offset = (max_peptides_per_dataset - len(peptide_order)) / 2
    xvals = np.arange(len(peptide_order)) + x_offset
    group_width = 0.72
    bar_width = group_width / len(single_context_order)
    context_colors = {
        'A3': ['#B7E4C7', '#40916C'],
        'DMF5': ['#BFD7EA', '#3F7FA8'],
    }.get(dataset_name, ['#BBBBBB', '#666666'])
    max_y = 0
    for context_idx, context in enumerate(single_context_order):
        yvals = pivot[context].to_numpy(dtype=int)
        max_y = max(max_y, int(yvals.max()) if len(yvals) else 0)
        xpos = xvals - group_width / 2 + bar_width / 2 + context_idx * bar_width
        ax.bar(
            xpos,
            yvals,
            color=context_colors[context_idx],
            edgecolor='black',
            linewidth=0.35,
            width=bar_width,
            label=single_context_legend_labels[context_idx],
        )
    ax.set_xticks(xvals)
    ax.set_xticklabels(peptide_order, rotation=45, ha='right')
    ax.set_xlim(-0.5, max_peptides_per_dataset - 0.5)
    ax.set_ylim(0, max(5, int(max_y * 1.55) + 1))
    ax.yaxis.set_major_formatter(count_formatter)
    ax.legend(frameon=True, loc='upper right', fontsize=10)
    ax.set_xlabel('Peptide', fontsize=AXIS_LABEL_SIZE)
    if col_idx == 0:
        ax.set_ylabel('Number of TCRs', fontsize=AXIS_LABEL_SIZE)
    style_axis(ax)


    # Row 5: TCR label multiplicity
    ax = axes[4, col_idx]
    subset = label_multiplicity_df[label_multiplicity_df['dataset'] == dataset_name].copy()
    counts_by_group = subset.set_index('label_multiplicity_group')['n_tcrs'] if not subset.empty else pd.Series(dtype=int)
    xvals = np.arange(len(label_multiplicity_order))
    yvals = np.array([counts_by_group.get(group, 0) for group in label_multiplicity_order], dtype=int)
    bar_colors = {
        'A3': ['#B7E4C7', '#74C69D', '#40916C', '#1B4332'],
        'DMF5': ['#BFD7EA', '#76A7C8', '#3F7FA8', '#12384F'],
    }.get(dataset_name, ['#DDDDDD', '#BBBBBB', '#888888', '#555555'])
    bars = ax.bar(
        xvals,
        yvals,
        color=bar_colors,
        edgecolor='black',
        linewidth=0.4,
        width=BAR_WIDTH,
    )
    annotate_bars(ax, bars, yvals)
    ax.set_xticks(xvals)
    ax.set_xticklabels(label_multiplicity_xticklabels)
    ax.set_xlim(-0.5, len(label_multiplicity_order) - 0.5)
    ax.set_ylim(0, max(5, int(yvals.max() * 1.22) + 1))
    ax.yaxis.set_major_formatter(count_formatter)
    ax.set_xlabel('TCR label multiplicity group', fontsize=AXIS_LABEL_SIZE)
    if col_idx == 0:
        ax.set_ylabel('Number of TCRs', fontsize=AXIS_LABEL_SIZE)
    style_axis(ax)

fig.suptitle('A3 vs DMF5 Dataset Analysis', fontsize=SUPTITLE_SIZE, y=0.995)
fig.tight_layout(rect=(0.03, 0.02, 1, 0.95), h_pad=8.0, w_pad=2.5)

row_titles = [
    'WT-distance distribution',
    'Pairwise TCR distance distribution',
    'Cross reactivity',
    'Peptide-level single-binder label context',
    'TCR label multiplicity (all loaded TCRs)',
]
for row_idx, title in enumerate(row_titles):
    row_axes = axes[row_idx, :]
    left = min(ax.get_position().x0 for ax in row_axes)
    right = max(ax.get_position().x1 for ax in row_axes)
    top = max(ax.get_position().y1 for ax in row_axes)
    fig.text(
        (left + right) / 2,
        top + 0.010,
        title,
        ha='center',
        va='bottom',
        fontsize=ROW_LABEL_SIZE,
        fontweight='bold',
    )

out_combined = output_dir / 'plot_00_combined_main_results.png'
plt.savefig(out_combined, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='white', transparent=False)
plt.show()
print('Saved:', out_combined)
print('Combined figure layout: 5 rows of plot types x 2 columns of datasets (A3 green, DMF5 blue).')


In [ ]:
# Validation printout for results section text
print('Validation summary (key reporting numbers):')
display(
    validation_df[
        [
            'dataset',
            'rows_analyzed',
            'unique_tcr_total',
            'unique_tcr_valid_len',
            'unique_tcr_invalid_len',
            'mean_distance_unique_tcr',
            'mean_distance_row_weighted',
            'crossreactive_binder_tcrs',
            'single_binder_context_tcrs',
            'label_multiplicity_unique_tcrs',
            'label_multiplicity_single_tcrs',
            'label_multiplicity_multiple_tcrs',
            'label_multiplicity_multiple_all_0_tcrs',
            'label_multiplicity_multiple_all_1_tcrs',
            'label_multiplicity_multiple_mixed_tcrs',
            'sampled_unique_tcrs',
            'pairs_computed',
            'mean_distance',
            'median_distance',
            'p05',
            'p95',
        ]
    ]
)

wt_invalid = validation_df[validation_df['unique_tcr_valid_len'] == 0]['dataset'].tolist()
if wt_invalid:
    print('WT-distance warning: these datasets have zero WT-valid-length TCRs and are excluded from WT-distance plotting:')
    print(', '.join(wt_invalid))

In [8]:
# This plot is included in the combined final figure above.
# Rerun cell 6 to regenerate plot_00_combined_main_results.png.


In [9]:
# This plot is included in the combined final figure above.
# Rerun cell 6 to regenerate plot_00_combined_main_results.png.


In [10]:
# The peptide-level single-binder quality plot is included as row 4 of the combined figure.
# The full-dataset TCR label multiplicity plot is included as row 5.
# The requested main result plots are generated together in cell 6.


## Optional Biological Extensions (Useful for Thesis Results)

These are additional biologically meaningful analyses you can add next:
1. **Binder vs non-binder TCR diversity**: compare unique TCR counts and clonality patterns between `label=1` and `label=0`.
2. **Peptide coverage long-tail**: rank peptides by number of unique binder TCRs to show immunodominance structure.
3. **WT-distance by binder status**: test whether binders are systematically closer/farther from WT than non-binders.
4. **Cross-reactivity normalization**: show both raw counts and percentages to compare datasets with different sizes.

### Runtime and Reproducibility Notes
- Analyses currently use `analysis_tcr_subsample_n=5000`, `subsample_n=250`, and `n_random_pairs=20000` for quick layout testing; set `analysis_tcr_subsample_n=None` and increase the pairwise settings for the final export.
- Randomness is controlled by `random_seed=42` for reproducible figures.
- All plots are exported to: `/cluster/project/reddy/katja/NGS_pipeline/results/thesis_dataset_analysis`